In [42]:
%matplotlib module://ipympl.backend_nbagg
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import random
from scipy.ndimage import maximum_filter

import sys
sys.path.insert(0, "/workspace/segmentation")
from tools import img_show3D, window, img_show2D, img_show3D_interactive, img_compare3D


In [2]:
array_3d = np.zeros((10,10,10))
shape = array_3d.shape

for z in range (shape[2]):

    for y in range (shape[1]):

        for x in range (shape[0]):

            array_3d[z, y, x] = random.randint(1, 100)

In [3]:
array_3d

array([[[ 55.,  43.,   2.,  99.,  62.,  56.,   4.,  18.,  15.,  16.],
        [ 29.,  10.,  18.,  91.,   9.,  15.,  86.,  28.,   4.,  17.],
        [  6.,  28.,  71.,  53.,  89.,  13.,  10.,  77.,  53.,  86.],
        [ 75.,   5.,  27.,  99.,  30.,  66.,  94.,  31.,  12.,  35.],
        [ 15.,  28.,   8.,  94.,   4.,  26.,  19.,  14.,  48.,  26.],
        [ 46.,  23.,   1.,  54.,  26., 100.,  24.,  13.,  83.,   1.],
        [ 39.,  19.,  77.,   1.,  11.,   4.,  61.,  75.,  74.,  91.],
        [ 67.,  31.,  70.,  28.,  76.,  55.,  82.,  10.,  77.,   7.],
        [ 24.,  40.,  48.,  80.,  94.,  94.,   2.,  72.,  56.,   9.],
        [ 77.,  81.,   9.,  92.,  49.,  19.,  73.,  38.,   6.,  29.]],

       [[ 50.,   5.,  27.,   9.,  34.,  66.,  39.,  56.,  68.,   3.],
        [ 56.,  48.,  21.,  75.,  22.,  49.,  66.,  35.,  78.,   6.],
        [ 31.,  28., 100.,  97.,  35.,   5.,  47.,  91.,  46.,  85.],
        [  9.,  34.,  59.,  12.,  18., 100.,  93.,  58.,   7.,  26.],
        [ 31.,  97

In [4]:
img_dir = Path ("/workspace/segmentation/data/practice/image_quadra/QUADRA_HC_021/retest_CT-AC.nii.gz")
img = sitk.ReadImage(img_dir)

mask_dir = Path ("/workspace/segmentation/data/practice/mask_quadra/QUADRA_HC_021/retest/liver.nii.gz")
mask = sitk.ReadImage(mask_dir)

In [5]:
img_syn = sitk.GetImageFromArray(array_3d)
img_syn.SetSpacing(img.GetSpacing())

print(img_syn.GetSpacing())

(1.5234375, 1.5234375, 2.0)


In [6]:
def cast(img):
    width, height, depth = img.GetSize()

    for y in range(height):
        for x in range(width):
            for z in range(depth):
                print(img[x, y, z], end="  ")
            print()  # Start a new line after each row.

In [ ]:
cast(img_syn)  

55.0  50.0  52.0  100.0  87.0  2.0  37.0  62.0  71.0  95.0  
43.0  5.0  87.0  31.0  19.0  86.0  40.0  37.0  98.0  32.0  
2.0  27.0  49.0  52.0  53.0  63.0  1.0  60.0  63.0  34.0  
99.0  9.0  70.0  24.0  85.0  91.0  74.0  94.0  17.0  78.0  
62.0  34.0  18.0  89.0  99.0  44.0  70.0  78.0  29.0  22.0  
56.0  66.0  48.0  29.0  88.0  16.0  16.0  4.0  33.0  58.0  
4.0  39.0  91.0  44.0  65.0  11.0  5.0  44.0  23.0  59.0  
18.0  56.0  53.0  87.0  23.0  45.0  86.0  38.0  63.0  92.0  
15.0  68.0  70.0  9.0  84.0  11.0  27.0  52.0  95.0  53.0  
16.0  3.0  88.0  96.0  1.0  16.0  99.0  98.0  32.0  24.0  
29.0  56.0  30.0  29.0  88.0  55.0  75.0  13.0  13.0  38.0  
10.0  48.0  74.0  11.0  31.0  18.0  87.0  74.0  77.0  57.0  
18.0  21.0  4.0  24.0  63.0  83.0  61.0  88.0  49.0  47.0  
91.0  75.0  93.0  87.0  41.0  87.0  40.0  29.0  24.0  89.0  
9.0  22.0  30.0  48.0  41.0  44.0  66.0  51.0  43.0  29.0  
15.0  49.0  80.0  86.0  83.0  35.0  76.0  78.0  42.0  21.0  
86.0  66.0  8.0  89.0  20.0  27.0  6

In [8]:
view = img_show3D_interactive(img_syn)

In [9]:
spacing = np.asarray(img_syn.GetSpacing())
radius_mm = 1
radius_voxels = np.maximum(np.ceil(radius_mm/spacing), 1)
kernel_size = tuple(int(2 * radius + 1) for radius in radius_voxels)


In [10]:
kernel_size

(3, 3, 3)

In [11]:
img_syn_array = sitk.GetArrayFromImage(img_syn)

In [12]:
maxima = maximum_filter(img_syn_array, size= kernel_size, mode= "nearest")

In [13]:
maxima

array([[[ 56.,  56.,  99.,  99.,  99.,  86.,  86.,  86.,  78.,  78.],
        [ 56., 100., 100., 100.,  99.,  89.,  91.,  91.,  91.,  86.],
        [ 75., 100., 100., 100., 100., 100., 100.,  94.,  91.,  86.],
        [ 97., 100., 100., 100., 100., 100., 100.,  94.,  91.,  86.],
        [ 97.,  97.,  99.,  99., 100., 100., 100.,  94.,  83.,  83.],
        [ 97.,  97.,  97.,  96., 100., 100., 100.,  89.,  91.,  91.],
        [ 82.,  96.,  96.,  96., 100., 100., 100.,  89.,  91.,  91.],
        [ 90.,  96.,  96.,  96.,  96.,  94.,  94.,  89.,  91.,  91.],
        [ 99.,  99.,  96.,  96.,  96.,  94.,  94.,  82.,  89.,  89.],
        [ 99.,  99.,  96.,  96.,  96.,  94.,  94.,  79.,  72.,  56.]],

       [[ 87.,  87.,  99.,  99.,  99.,  91.,  99.,  99.,  99.,  88.],
        [ 87., 100., 100., 100., 100., 100., 100.,  99.,  99.,  88.],
        [ 91., 100., 100., 100., 100., 100., 100.,  99.,  99.,  94.],
        [ 97., 100., 100., 100., 100., 100., 100.,  94., 100., 100.],
        [ 97.,  97

In [19]:
coordinates = np.argwhere(
    img_syn_array == maxima
)

In [40]:
selected_point_mask = np.zeros((10,10,10))

for coordinate in coordinates:
    selected_point_mask[tuple(coordinate)] = 1


selected_point_mask

array([[[0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]

In [41]:
selected = sitk.GetImageFromArray(selected_point_mask)

In [44]:
compare = img_compare3D (
    selected,
    img_syn,
    name1= "selected",
    name2= "original image"
)